In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,MinMaxScaler,PowerTransformer
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn import set_config
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score,mean_absolute_error

import optuna
import dagshub
import mlflow

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load data

df = pd.read_csv(r'C:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\notebooks\data_eda.csv')
df.sample(2)

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
18794,INDORES08DEL01,37.0,4.7,22.725748,75.898497,22.765747,75.938497,2022-04-03,fog,high,...,INDO,3,4,sunday,1,5.0,15.0,afternoon,6.050406,medium
9311,KNPRES16DEL01,25.0,4.7,26.482581,80.315628,26.492581,80.325628,2022-02-11,stormy,low,...,KNP,11,2,friday,0,5.0,10.0,morning,1.492284,short


In [3]:
set_config(transform_output='pandas')

In [4]:
df.isnull().sum()

rider_id                   0
age                     1854
ratings                 1908
restaurant_latitude     3630
restaurant_longitude    3630
delivery_latitude       3630
delivery_longitude      3630
order_date                 0
weather                  525
traffic                  510
vehicle_condition          0
type_of_order              0
type_of_vehicle            0
multiple_deliveries      993
festival                 228
city_type               1198
time_taken                 0
city_name                  0
order_day                  0
order_month                0
order_day_of_week          0
is_weekend                 0
pickup_time_minutes     1640
order_time_hour         1640
order_time_of_day       2070
distance                3630
distance_type           3630
dtype: int64

In [5]:
# dropping columns
df.drop(columns=['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"],
                    inplace=True)

df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
30271,33.0,4.7,sunny,low,2,meal,scooter,1.0,no,urban,29,0,15.0,night,19.761066,very_long
1939,32.0,4.4,sandstorms,jam,1,meal,scooter,0.0,no,metropolitian,33,0,15.0,evening,4.657620,short


# Model Building

In [6]:
temp_df = df.copy().dropna()

In [7]:
temp_df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
16269,23.0,4.7,stormy,high,1,buffet,motorcycle,0.0,no,urban,20,1,5.0,morning,1.489846,short
41088,26.0,5.0,fog,medium,2,buffet,electric_scooter,1.0,no,urban,32,1,10.0,evening,10.906471,long


In [8]:
temp_df.isnull().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [9]:
X = temp_df.drop(columns=['time_taken'])
y = temp_df['time_taken']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape,y_test.shape

((30156, 15), (7539,))

In [10]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [11]:
# Transforming target column

pt = PowerTransformer(method='yeo-johnson')

y_train_trans = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_trans = pt.transform(y_test.values.reshape(-1,1))

y_train_trans


,x0
0,2.028672
1,0.554539
2,-2.024267
3,-0.173699
4,0.554539
...,...
30151,0.457580
30152,-0.173699
30153,-1.350937
30154,0.047111


In [12]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [13]:
# build a preprocessor

preprocessor = ColumnTransformer(
    transformers=
    [
        ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False), nominal_cat_cols),
        ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],handle_unknown="use_encoded_value",unknown_value=-1), ordinal_cat_cols)
    ],
        remainder="passthrough",n_jobs=-1,verbose_feature_names_out=False
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [14]:
# build the pipeline

processing_pipeline = Pipeline(steps=[
                                ("preprocess",preprocessor)
                            ])

processing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [15]:
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

## Optuna

In [16]:
dagshub.init(repo_owner='jay-kanakia', repo_name='swiggy_delivery_time_prediction', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow')
mlflow.set_experiment(experiment_name='Exp 3 : Optuna LightGBM HP')

Accessing as jay-kanakia

Initialized MLflow to track repo "jay-kanakia/swiggy_delivery_time_prediction"

Repository jay-kanakia/swiggy_delivery_time_prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/cafda3023e6e469abe8b2a835bb63d03', creation_time=1775392161345, experiment_id='4', last_update_time=1775392161345, lifecycle_stage='active', name='Exp 3 : Optuna LightGBM HP', tags={}, workspace='default'>

In [17]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            'n_estimator' : trial.suggest_int('n_estimator',10,200),
            'max_depth' : trial.suggest_int('max_depth',1,40),
            'learning_rate' :  trial.suggest_float('learning_rate',0.01,0.3),
            'subsample' :  trial.suggest_float('subsample',0.5,1),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            'min_sum_hessian_in_leaf' :  trial.suggest_int('min_child_weight',5,20),
            'min_split_gain' : trial.suggest_float('min_split_gain',0,10),
            'reg_lambda' : trial.suggest_float('reg_lambda',0,100),
            'random_state' : 42,
            'n_jobs' : -1
        }

        # log model parameters
        mlflow.log_params(params)

        lgbm = LGBMRegressor(**params)

        model = TransformedTargetRegressor(regressor=lgbm,transformer=pt)

        model.fit(X_train_trans,y_train)

        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)

        mlflow.log_metric('Traning accuracy',r2_score(y_train,y_pred_train))
        mlflow.log_metric('Testing accuracy',r2_score(y_test,y_pred_test))

        score = cross_val_score(model,X_train_trans,y_train,cv=5,scoring='neg_mean_absolute_error',n_jobs=-1)

        mean_score = -(score.mean())

        mlflow.log_metric('cross_val_error',mean_score)

        return mean_score

    

In [19]:
# create study object

study = optuna.create_study(direction='minimize',study_name='LGBM HP')

with mlflow.start_run(run_name='Best Model'):
    
    # optimize the objective function
    study.optimize(objective,n_trials=50,n_jobs=-1,show_progress_bar=True)

    # log best parameters
    mlflow.log_params(study.best_params)

    # train the model on best parameters
    best_lgbm = LGBMRegressor(**study.best_params)

    model = TransformedTargetRegressor(regressor=best_lgbm,transformer=pt)

    model.fit(X_train_trans,y_train)

    # get the prediction
    y_pred_train = model.predict(X_train_trans)
    y_pred_test = model.predict(X_test_trans)

    scores = cross_val_score(model,X_train_trans,y_train,scoring='neg_mean_absolute_error',cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_lgbm,artifact_path='model')

[I 2026-04-05 18:23:01,769] A new study created in memory with name: LGBM HP
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run merciful-jay-632 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/cfbc4384521947f792911bba54377ac8
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4
🏃 View run wise-grub-774 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/a63e09c3c9724ee5aa9b695eca27f8c2
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4
🏃 View run chill-snipe-806 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/bd4cadb47f6341f8b3403aa61a65203d
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4
🏃 View run incongruous-crow-585 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/162d31d99ee4443fbfc52149bbe4720a
🧪 View experiment at: https://dagshub.com/jay

Best trial: 7. Best value: 3.69024:   2%|▏         | 1/50 [01:01<50:29, 61.82s/it]

[I 2026-04-05 18:24:05,493] Trial 7 finished with value: 3.690242655790324 and parameters: {'n_estimator': 191, 'max_depth': 6, 'learning_rate': 0.26687280091297094, 'subsample': 0.8372687850315467, 'num_leaves': 57, 'min_child_weight': 9, 'min_split_gain': 7.898421704106909, 'reg_lambda': 20.354639819327236}. Best is trial 7 with value: 3.690242655790324.
🏃 View run sincere-slug-269 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/0f72aa14149240ce9915022b764da07b
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4


Best trial: 2. Best value: 3.5328:   4%|▍         | 2/50 [01:03<21:17, 26.61s/it] 

...

[I 2026-04-05 18:28:43,505] Trial 45 finished with value: 3.3334875341490653 and parameters: {'n_estimator': 40, 'max_depth': 10, 'learning_rate': 0.293314854531882, 'subsample': 0.5051324377019635, 'num_leaves': 139, 'min_child_weight': 13, 'min_split_gain': 0.2748051376025549, 'reg_lambda': 54.79728324305178}. Best is trial 48 with value: 3.098601299582346.


c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but PowerTransformer was fitted without feature names
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Warning] Unknown parameter: n_estimator
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002642 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 352
[LightGBM] [Info] Number of data points in the train set: 30156, number of used features: 25
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

2026/04/05 18:28:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 18:29:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best Model at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4/runs/3db03d2ee71d41c4af16b5f7cf395c47
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/4


In [21]:
study.best_params

{'n_estimator': 40,
 'max_depth': 11,
 'learning_rate': 0.18024722308848545,
 'subsample': 0.7972018508352986,
 'num_leaves': 118,
 'min_child_weight': 14,
 'min_split_gain': 0.007311476316166993,
 'reg_lambda': 54.73782637648871}

In [22]:
study.best_value

3.098601299582346

In [23]:
optuna.visualization.plot_optimization_history(study)

In [25]:
optuna.visualization.plot_param_importances(study)

In [26]:
optuna.visualization.plot_slice(study)